In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.layers import Bidirectional
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# **Read data**

In [2]:
df = pd.read_csv('IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# **Describe data**

In [3]:
df.describe().T

,count,unique,top,freq
review,50000,49582,Loved today's show!!! It was a variety and not...,5
sentiment,50000,2,positive,25000


In [4]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [5]:
df.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)

/tmp/ipykernel_571/1137712857.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)


# **Preprocess data**

In [6]:
def remove_tags(str):
  #remove HTML tags
  res = re.sub(r'<[^>]+>', '', str)

  #remove URLs
  res = re.sub(r'https?://\S+', '', res)

  #remove non-alphanumeric characters
  res = re.sub(r'[^a-zA-Z0-9' + r'\s]', '', res)

  #convert to lower case
  res = res.lower()
  return res

In [7]:
df['review'] = df['review'].apply(remove_tags)

In [8]:
# nltk.download('stopwords')

# from nltk.corpus import stopwords

# stop_words = set(stopwords.words('english'))

# negative_words = {'no', 'not', 'nor', 'don', "don't", 'ain', 'aren', "aren't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't", 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"}
# stop_words = stop_words - negative_words

# df['review'] = df['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))

In [9]:
#nltk.download('wordnet')
#w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
#lemmatizer = nltk.stem.WordNetLemmatizer()
#def lemmatize_text(text):
#    st = ""
#    for w in w_tokenizer.tokenize(text):
#        st = st + lemmatizer.lemmatize(w) + " "
#    return st
#df['review'] = df.review.apply(lemmatize_text)

In [10]:
# split data into training data and test data
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42, stratify=df['sentiment'])

# **Tokenization**

In [11]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production the filming tech...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically theres a family where a little boy j...,0
4,petter matteis love in the time of money is a ...,1


In [12]:
# Tokenize text data
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(train_data["review"])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=500)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=500)

In [13]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

# **Build model**

In [14]:
from tensorflow.keras.layers import GlobalMaxPooling1D, Dropout

model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=128))
model.add(Bidirectional(LSTM(64, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)))
model.add(GlobalMaxPooling1D())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation="sigmoid"))

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
# compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# **Train model**

In [16]:
from tensorflow.keras.callbacks import EarlyStopping

# Define the EarlyStopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model with early stopping
model.fit(X_train, Y_train, epochs=10, batch_size=64, validation_split=0.2, callbacks=[early_stopping])

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 674s 1s/step - accuracy: 0.7958 - loss: 0.4245 - val_accuracy: 0.8808 - val_loss: 0.2814
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 674s 1s/step - accuracy: 0.9054 - loss: 0.2459 - val_accuracy: 0.8777 - val_loss: 0.2985
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 692s 1s/step - accuracy: 0.9310 - loss: 0.1884 - val_accuracy: 0.8874 - val_loss: 0.2767
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 675s 1s/step - accuracy: 0.9503 - loss: 0.1417 - val_accuracy: 0.8878 - val_loss: 0.3068
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 703s 1s/step - accuracy: 0.9647 - loss: 0.1046 - val_accuracy: 0.8841 - val_loss: 0.3319
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 713s 1s/step - accuracy: 0.9758 - loss: 0.0740 - val_accuracy: 0.8841 - val_loss: 0.3569


# **Prediction**

In [17]:
from sklearn.metrics import classification_report
predicted_probabilities = model.predict(X_test)
predicted = (predicted_probabilities > 0.5).astype(int)
print(classification_report(Y_test, predicted))

313/313 ━━━━━━━━━━━━━━━━━━━━ 65s 206ms/step
              precision    recall  f1-score   support

           0       0.89      0.90      0.89      5000
           1       0.90      0.89      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

